In [0]:
from pyspark.sql import functions as F

CATALOG      = "workspace"
SILVER_TABLE = f"{CATALOG}.sme_risk_silver.companies"
GOLD_SCHEMA  = f"{CATALOG}.sme_risk_gold"

## dim_company
One row per company (ACN). Descriptive attributes only; status, classification and dates live in their own dimensions or in the facts.
Key: company_key = ACN as a number.

In [0]:
query = f"""
SELECT
    CAST(acn AS BIGINT)                       AS company_key,
    acn,
    company_name,
    registration_date,
    abn,
    has_abn,
    is_acn_named,
    former_name_count,
    former_name_count > 0               AS has_name_change,
    previous_state,
    previous_state_label,
    state_registration_number,
    company_name_has_export_artifact,
    _snapshot_date                      AS as_of_snapshot_date
FROM {SILVER_TABLE}
"""
df = spark.sql(query)

In [0]:
df.limit(10).display()

In [0]:
(df.write
   .mode("overwrite")
   .format("delta")
   .option("overwriteSchema", "true")
   .saveAsTable(f"{GOLD_SCHEMA}.dim_company"))

In [0]:
%sql
SELECT
    (SELECT COUNT(*) FROM workspace.sme_risk_silver.companies)  AS silver_rows,      -- expect 4,016,332
    COUNT(*)                                                    AS dim_rows,         -- must equal silver_rows
    COUNT(DISTINCT company_key)                                 AS distinct_keys,    -- must equal dim_rows (no duplicate keys)
    COUNT(DISTINCT acn)                                         AS distinct_acns,    -- must equal dim_rows
    COUNT_IF(company_key IS NULL)                               AS null_keys,        -- expect 0
    COUNT_IF(has_abn)                                           AS with_abn,         -- expect 2,385,168
    COUNT_IF(has_name_change)                                   AS with_name_change,
    COUNT_IF(is_acn_named)                                      AS acn_named
FROM workspace.sme_risk_gold.dim_company

## dim_company_classification
One row per type + class + sub-class combination found in Silver, with ASIC labels and the **borrower segment** (agreed 2026-10-13).
Key: `classification_key` = the three codes joined (e.g. `APTY-LMSH-PROP`): readable and the same every week.
Segment rules apply in order: foreign / registrable / CCIV by type first, then super trustee, home unit, not-for-profit by sub-class, then proprietary = Trading, remaining public = Other public.

In [0]:
query = f"""
WITH combos AS (
    SELECT DISTINCT
        company_type,      company_type_label,
        company_class,     company_class_label,
        company_sub_class, company_sub_class_label
    FROM {SILVER_TABLE}
)
SELECT
    concat_ws('-', company_type, company_class, company_sub_class) AS classification_key,
    company_type,      company_type_label,
    company_class,     company_class_label,
    company_sub_class, company_sub_class_label,
    CASE
        WHEN company_type = 'FNOS'                         THEN 'Foreign company'
        WHEN company_type = 'RACN'                         THEN 'Registrable body'
        WHEN company_type = 'CCIV'                         THEN 'CCIV'
        WHEN company_sub_class = 'PSTC'                    THEN 'Super trustee'
        WHEN company_sub_class = 'HUNT'                    THEN 'Home unit'
        WHEN company_sub_class IN ('ULSN', 'PNPC', 'NLTD') THEN 'Not-for-profit'
        WHEN company_type = 'APTY'                         THEN 'Trading company'
        WHEN company_type = 'APUB'                         THEN 'Other public'
        ELSE 'Unclassified'
    END AS borrower_segment,
    CASE
        WHEN company_type = 'FNOS'                         THEN 6
        WHEN company_type = 'RACN'                         THEN 7
        WHEN company_type = 'CCIV'                         THEN 8
        WHEN company_sub_class = 'PSTC'                    THEN 2
        WHEN company_sub_class = 'HUNT'                    THEN 4
        WHEN company_sub_class IN ('ULSN', 'PNPC', 'NLTD') THEN 3
        WHEN company_type = 'APTY'                         THEN 1
        WHEN company_type = 'APUB'                         THEN 5
        ELSE 99
    END AS segment_sort_order
FROM combos
"""
df = spark.sql(query)

In [0]:
df.orderBy("segment_sort_order", "classification_key").display()

In [0]:
(df.write
   .mode("overwrite")
   .format("delta")
   .option("overwriteSchema", "true")
   .saveAsTable(f"{GOLD_SCHEMA}.dim_company_classification"))

In [0]:
%sql
SELECT
    COUNT(*)                                   AS dim_rows,
    COUNT(DISTINCT classification_key)         AS distinct_keys,      -- must equal dim_rows
    COUNT_IF(borrower_segment = 'Unclassified') AS unclassified        -- expect 0
FROM workspace.sme_risk_gold.dim_company_classification

In [0]:
%sql
SELECT
    d.segment_sort_order,
    COALESCE(d.borrower_segment, 'NO MATCH')   AS borrower_segment,   -- 'NO MATCH' must not appear
    COUNT(*)                                   AS companies,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM workspace.sme_risk_silver.companies s
LEFT JOIN workspace.sme_risk_gold.dim_company_classification d
  ON concat_ws('-', s.company_type, s.company_class, s.company_sub_class) = d.classification_key
GROUP BY d.segment_sort_order, d.borrower_segment
ORDER BY d.segment_sort_order

## dim_status
One row per ASIC status code found in Silver.
Key: `status_key` = the status code (e.g. `REGD`): readable and the same every week.
Flags use the same rules as Silver: active = REGD, SOFF, EXAD, NOAC · distressed = SOFF, EXAD · deregistered = DRGD · population excludes CNCL, DISS.

In [0]:
query = f"""
WITH statuses AS (
    SELECT DISTINCT status_code, status_label
    FROM {SILVER_TABLE}
)
SELECT
    status_code                                          AS status_key,
    status_code,
    status_label,
    status_code IN ('REGD', 'SOFF', 'EXAD', 'NOAC')      AS is_active,
    status_code IN ('SOFF', 'EXAD')                      AS is_distressed,
    status_code = 'DRGD'                                 AS is_deregistered,
    status_code NOT IN ('CNCL', 'DISS')                  AS in_population,
    CASE
        WHEN status_code = 'REGD'               THEN 'Registered'
        WHEN status_code IN ('SOFF', 'EXAD')    THEN 'Distressed'
        WHEN status_code = 'NOAC'               THEN 'Not active'
        WHEN status_code = 'DRGD'               THEN 'Deregistered'
        ELSE 'Excluded'
    END                                                  AS status_group,
    CASE status_code
        WHEN 'REGD' THEN 1  WHEN 'SOFF' THEN 2  WHEN 'EXAD' THEN 3
        WHEN 'NOAC' THEN 4  WHEN 'DRGD' THEN 5  ELSE 9
    END                                                  AS status_sort_order
FROM statuses
"""
df = spark.sql(query)

In [0]:
(df.write
   .mode("overwrite")
   .format("delta")
   .option("overwriteSchema", "true")
   .saveAsTable(f"{GOLD_SCHEMA}.dim_status"))

In [0]:
%sql
SELECT
    COUNT(*)                    AS dim_rows,         -- expect 6 (REGD, DRGD, SOFF, EXAD, NOAC, DISS)
    COUNT(DISTINCT status_key)  AS distinct_keys,    -- must equal dim_rows
    COUNT_IF(status_label IS NULL) AS missing_labels -- expect 0
FROM workspace.sme_risk_gold.dim_status

In [0]:
%sql
SELECT
    d.status_sort_order,
    COALESCE(d.status_key, 'NO MATCH')                       AS status_key,
    d.status_group,
    COUNT(*)                                                 AS companies,
    COUNT_IF(s.is_active       <> d.is_active)               AS active_mismatch,       -- expect 0
    COUNT_IF(s.is_distressed   <> d.is_distressed)           AS distressed_mismatch,   -- expect 0
    COUNT_IF(s.is_deregistered <> d.is_deregistered)         AS deregistered_mismatch, -- expect 0
    COUNT_IF(s.in_population   <> d.in_population)           AS population_mismatch    -- expect 0
FROM workspace.sme_risk_silver.companies s
LEFT JOIN workspace.sme_risk_gold.dim_status d
  ON s.status_code = d.status_key
GROUP BY d.status_sort_order, d.status_key, d.status_group
ORDER BY d.status_sort_order

## dim_age_band
Fixed list of company age bands (KPI spec v1): yearly bands for the first 5 years, then 5–10 and 10+, plus Unknown for the 4 companies without a registration date.
Bands run from `min_age_years` (included) to `max_age_years` (excluded). Age is counted in full years from the registration date (anniversaries).
Key: `age_band_key` (1–7, 0 = Unknown): fixed, never changes.

In [0]:
query = """
WITH ages AS (
    SELECT explode(sequence(0, 10)) AS age_years      -- ages 0..10 (10 covers 10+)
    UNION ALL
    SELECT NULL                                       -- missing registration date
),
bands AS (
    SELECT DISTINCT
        CASE
            WHEN age_years IS NULL THEN 0
            WHEN age_years < 5     THEN age_years + 1   -- 0 → 1 … 4 → 5
            WHEN age_years < 10    THEN 6
            ELSE 7
        END AS age_band_key
    FROM ages
)
SELECT
    age_band_key,

    CASE
        WHEN age_band_key = 0 THEN 'Unknown'
        WHEN age_band_key = 6 THEN '5-10 years'
        WHEN age_band_key = 7 THEN '10+ years'
        ELSE concat(age_band_key - 1, '-', age_band_key, ' years')
    END AS age_band_label,

    CASE
        WHEN age_band_key = 0 THEN NULL
        WHEN age_band_key = 6 THEN 5
        WHEN age_band_key = 7 THEN 10
        ELSE age_band_key - 1
    END AS min_age_years,

    CASE
        WHEN age_band_key IN (0, 7) THEN NULL
        WHEN age_band_key = 6 THEN 10
        ELSE age_band_key
    END AS max_age_years,

    CASE
        WHEN age_band_key = 0 THEN 'Unknown'
        WHEN age_band_key BETWEEN 1 AND 5 THEN 'New (under 5 years)'
        WHEN age_band_key = 6 THEN 'Established (5-10 years)'
        ELSE 'Mature (10+ years)'
    END AS age_group,

    age_band_key = 1 AS is_first_year
FROM bands
"""
df = spark.sql(query)

In [0]:
(df.write
   .mode("overwrite")
   .format("delta")
   .option("overwriteSchema", "true")
   .saveAsTable(f"{GOLD_SCHEMA}.dim_age_band"))

In [0]:
%sql
WITH ordered AS (
    SELECT *, LEAD(min_age_years) OVER (ORDER BY min_age_years) AS next_min
    FROM workspace.sme_risk_gold.dim_age_band
    WHERE age_band_key <> 0
)
SELECT
    (SELECT COUNT(*) FROM workspace.sme_risk_gold.dim_age_band)                      AS dim_rows,       -- expect 8
    (SELECT COUNT(DISTINCT age_band_key) FROM workspace.sme_risk_gold.dim_age_band)  AS distinct_keys,  -- expect 8
    COUNT_IF(next_min IS NOT NULL AND max_age_years <> next_min)                      AS gaps_or_overlaps, -- expect 0
    COUNT_IF(next_min IS NULL AND max_age_years IS NOT NULL)                          AS last_band_not_open -- expect 0
FROM ordered

In [0]:
%sql
WITH ages AS (
    SELECT
        acn,
        FLOOR(MONTHS_BETWEEN(_snapshot_date, registration_date) / 12) AS age_years
    FROM workspace.sme_risk_silver.companies
    WHERE status_code <> 'DRGD'          -- companies still on the register
)
SELECT
    COALESCE(b.age_band_key, -1)   AS age_band_key,    -- -1 would mean "no band found": must not appear
    b.age_band_label,
    COUNT(*)                       AS companies,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM ages a
LEFT JOIN workspace.sme_risk_gold.dim_age_band b
  ON (a.age_years IS NULL AND b.age_band_key = 0)
  OR (a.age_years >= b.min_age_years AND (a.age_years < b.max_age_years OR b.max_age_years IS NULL))
GROUP BY b.age_band_key, b.age_band_label
ORDER BY b.age_band_key

In [0]:
df.orderBy("age_band_key").display()

## dim_date
One row per calendar day, from 1 January of the earliest registration year (1817) to 31 December of the snapshot year, built from Silver so the range grows automatically.
Includes the Australian financial year (July–June), since registrations peak around 30 June.
Key: `date_key` = YYYYMMDD as a number (e.g. 20261012). `date_key = 0` is the "No date" row for missing dates (not deregistered / no registration date).

In [0]:
query = f"""
WITH bounds AS (
    SELECT
        make_date(year(MIN(registration_date)), 1, 1)   AS start_date,
        make_date(year(MAX(_snapshot_date)), 12, 31)    AS end_date
    FROM {SILVER_TABLE}
),
days AS (
    SELECT explode(sequence(start_date, end_date, INTERVAL 1 DAY)) AS d
    FROM bounds
),
calendar AS (
    SELECT
        CAST(date_format(d, 'yyyyMMdd') AS INT)                   AS date_key,
        d                                                         AS full_date,
        year(d)                                                   AS year,
        quarter(d)                                                AS quarter,
        month(d)                                                  AS month,
        date_format(d, 'MMMM')                                    AS month_name,
        date_format(d, 'yyyy-MM')                                 AS year_month,
        trunc(d, 'MM')                                            AS month_start_date,
        dayofmonth(d)                                             AS day_of_month,
        date_format(d, 'EEEE')                                    AS day_name,
        dayofweek(d) IN (1, 7)                                    AS is_weekend,
        CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END AS financial_year,
        concat('FY', CASE WHEN month(d) >= 7 THEN year(d) + 1 ELSE year(d) END) AS financial_year_label,
        ((month(d) + 5) % 12) div 3 + 1                           AS financial_quarter,
        month(d) = 6 AND dayofmonth(d) = 30                       AS is_financial_year_end
    FROM days
)
SELECT * FROM calendar
UNION ALL
SELECT
    0, NULL, NULL, NULL, NULL, 'No date', 'No date', NULL, NULL, 'No date',
    NULL, NULL, 'No date', NULL, NULL
"""
df = spark.sql(query)

In [0]:
df.filter("full_date BETWEEN '2026-06-28' AND '2026-07-02' OR date_key = 0").orderBy("date_key").display()

In [0]:
(df.write
   .mode("overwrite")
   .format("delta")
   .option("overwriteSchema", "true")
   .saveAsTable(f"{GOLD_SCHEMA}.dim_date"))

In [0]:
%sql
SELECT
    COUNT(*)                                             AS dim_rows,
    COUNT(DISTINCT date_key)                             AS distinct_keys,    -- must equal dim_rows
    MIN(full_date)                                       AS first_date,       -- expect 1817-01-01
    MAX(full_date)                                       AS last_date,        -- expect 2026-12-31
    DATEDIFF(MAX(full_date), MIN(full_date)) + 2         AS expected_rows,    -- all days + the 'No date' row; must equal dim_rows
    COUNT_IF(date_key = 0)                               AS no_date_rows      -- expect 1
FROM workspace.sme_risk_gold.dim_date

In [0]:
%sql
SELECT
    COUNT_IF(s.registration_date IS NOT NULL AND r.date_key IS NULL)    AS reg_dates_missing,    -- expect 0
    COUNT_IF(s.deregistration_date IS NOT NULL AND d.date_key IS NULL)  AS dereg_dates_missing,  -- expect 0
    COUNT_IF(n.date_key IS NULL)                                        AS snapshot_date_missing -- expect 0
FROM workspace.sme_risk_silver.companies s
LEFT JOIN workspace.sme_risk_gold.dim_date r ON r.full_date = s.registration_date
LEFT JOIN workspace.sme_risk_gold.dim_date d ON d.full_date = s.deregistration_date
LEFT JOIN workspace.sme_risk_gold.dim_date n ON n.full_date = s._snapshot_date

## fct_company_snapshot
Grain: one row per company per snapshot. Grain: one row per company for the latest snapshot. Rebuilt from Silver each run.
Keys are built with the same rules as the dimensions (no joins needed): company = ACN as number · classification = type-class-sub_class · status = status code · age band = age rule · dates = YYYYMMDD (0 = no date).
Age is measured at the snapshot date, or at the deregistration date for deregistered companies (their age when they left the register).
Measures are 0/1 so they can be summed: company_count, is_active, is_distressed, is_deregistered, in_population.

In [0]:
query = f"""
WITH base AS (
    SELECT
        *,
        CASE WHEN status_code = 'DRGD' THEN deregistration_date ELSE _snapshot_date END AS age_as_of_date
    FROM {SILVER_TABLE}
),
aged AS (
    SELECT
        *,
        CAST(FLOOR(MONTHS_BETWEEN(age_as_of_date, registration_date) / 12) AS INT) AS age_years
    FROM base
)
SELECT
    -- keys
    CAST(acn AS BIGINT)                                                       AS company_key,
    concat_ws('-', company_type, company_class, company_sub_class)            AS classification_key,
    status_code                                                               AS status_key,
    CASE
        WHEN age_years IS NULL THEN 0
        WHEN age_years < 5     THEN age_years + 1
        WHEN age_years < 10    THEN 6
        ELSE 7
    END                                                                       AS age_band_key,
    COALESCE(CAST(date_format(registration_date,   'yyyyMMdd') AS INT), 0)    AS registration_date_key,
    COALESCE(CAST(date_format(deregistration_date, 'yyyyMMdd') AS INT), 0)    AS deregistration_date_key,
    CAST(date_format(_snapshot_date, 'yyyyMMdd') AS INT)                      AS snapshot_date_key,

    -- measures
    age_years,
    1                                AS company_count,
    CAST(is_active       AS INT)     AS is_active,
    CAST(is_distressed   AS INT)     AS is_distressed,
    CAST(is_deregistered AS INT)     AS is_deregistered,
    CAST(in_population   AS INT)     AS in_population
FROM aged
"""
df = spark.sql(query)

In [0]:
df.limit(10).display()

In [0]:
(df.write
   .mode("overwrite")
   .format("delta")
   .option("overwriteSchema", "true")
   .saveAsTable(f"{GOLD_SCHEMA}.fct_company_snapshot"))

In [0]:
%sql
SELECT
    COUNT(*)                                          AS fact_rows,             -- 4,016,332 per snapshot
    COUNT(DISTINCT f.company_key, f.snapshot_date_key)    AS distinct_grain,        -- must equal fact_rows
    COUNT_IF(c.company_key IS NULL)                   AS orphan_company,        -- expect 0
    COUNT_IF(k.classification_key IS NULL)            AS orphan_classification, -- expect 0
    COUNT_IF(s.status_key IS NULL)                    AS orphan_status,         -- expect 0
    COUNT_IF(a.age_band_key IS NULL)                  AS orphan_age_band,       -- expect 0
    COUNT_IF(r.date_key IS NULL)                      AS orphan_reg_date,       -- expect 0
    COUNT_IF(d.date_key IS NULL)                      AS orphan_dereg_date,     -- expect 0
    COUNT_IF(n.date_key IS NULL)                      AS orphan_snapshot_date   -- expect 0
FROM workspace.sme_risk_gold.fct_company_snapshot f
LEFT JOIN workspace.sme_risk_gold.dim_company                c ON f.company_key             = c.company_key
LEFT JOIN workspace.sme_risk_gold.dim_company_classification k ON f.classification_key      = k.classification_key
LEFT JOIN workspace.sme_risk_gold.dim_status                 s ON f.status_key              = s.status_key
LEFT JOIN workspace.sme_risk_gold.dim_age_band               a ON f.age_band_key            = a.age_band_key
LEFT JOIN workspace.sme_risk_gold.dim_date                   r ON f.registration_date_key   = r.date_key
LEFT JOIN workspace.sme_risk_gold.dim_date                   d ON f.deregistration_date_key = d.date_key
LEFT JOIN workspace.sme_risk_gold.dim_date                   n ON f.snapshot_date_key       = n.date_key

In [0]:
%sql
SELECT
    snapshot_date_key,
    SUM(company_count)                      AS companies,      -- 4,016,332
    SUM(is_active)                          AS active,         -- 3,804,740 (REGD + SOFF + EXAD + NOAC)
    SUM(is_distressed)                      AS distressed,     -- 74,941 (SOFF + EXAD)
    SUM(is_deregistered)                    AS deregistered,   -- 211,591
    SUM(in_population)                      AS in_population,  -- 4,016,331 (all except 1 DISS)
    COUNT_IF(age_band_key = 0)              AS unknown_age,    -- 4 (3 NOAC + 1 DRGD)
    COUNT_IF(age_years < 0)                 AS negative_age    -- expect 0
FROM workspace.sme_risk_gold.fct_company_snapshot
GROUP BY snapshot_date_key
ORDER BY snapshot_date_key

## fct_exposure
Grain: one row per company × age band × snapshot. Holds the time each company was at risk inside the observation window (exposure) and whether it deregistered — the inputs for the survival KPIs (KPI spec v1).
Window: earliest deregistration date in the file → snapshot date (ASIC keeps one year of deregistrations).
At risk from max(window start, registration date) until deregistration or window end; time split at anniversaries (add_months) into age bands; a deregistration counts in the band of the company's age on its deregistration date.
Excluded: not in population (DISS) and the 4 companies without a registration date.

In [0]:
query = f"""
WITH obs_window AS (
    SELECT
        MIN(deregistration_date) AS window_start,
        MAX(_snapshot_date)      AS window_end
    FROM {SILVER_TABLE}
),
spells AS (                                          -- one at-risk period per company
    SELECT
        s.acn, s.company_type, s.company_class, s.company_sub_class, s.status_code,
        s.registration_date, s.is_deregistered,
        w.window_start, w.window_end,
        GREATEST(w.window_start, s.registration_date)                                  AS entry_date,
        CASE WHEN s.is_deregistered THEN s.deregistration_date ELSE w.window_end END   AS exit_date
    FROM {SILVER_TABLE} s
    CROSS JOIN obs_window w
    WHERE s.in_population
      AND s.registration_date IS NOT NULL
),
valid AS (
    SELECT
        *,
        CAST(FLOOR(MONTHS_BETWEEN(entry_date, registration_date) / 12) AS INT) AS age_at_entry,
        CAST(FLOOR(MONTHS_BETWEEN(exit_date,  registration_date) / 12) AS INT) AS age_at_exit
    FROM spells
    WHERE exit_date > entry_date OR is_deregistered  -- drop companies registered on the last day (no time at risk)
),
pieces AS (                                          -- split each period at birthdays
    SELECT
        v.*,
        k                                                                       AS age_years,
        GREATEST(entry_date, add_months(registration_date, 12 * k))             AS piece_start,
        LEAST(exit_date,     add_months(registration_date, 12 * (k + 1)))       AS piece_end
    FROM valid v
    LATERAL VIEW explode(sequence(age_at_entry, age_at_exit)) e AS k
),
banded AS (
    SELECT
        acn, company_type, company_class, company_sub_class, status_code,
        window_start, window_end,
        CASE
            WHEN age_years < 5  THEN age_years + 1
            WHEN age_years < 10 THEN 6
            ELSE 7
        END                                                                     AS age_band_key,
        DATEDIFF(piece_end, piece_start)                                        AS exposure_days,
        CASE WHEN is_deregistered AND age_years = age_at_exit THEN 1 ELSE 0 END AS deregistrations
    FROM pieces
    WHERE DATEDIFF(piece_end, piece_start) > 0
       OR (is_deregistered AND age_years = age_at_exit)  -- keep the event even if it falls exactly on a birthday
)
SELECT
    CAST(acn AS BIGINT)                                              AS company_key,
    concat_ws('-', company_type, company_class, company_sub_class)   AS classification_key,
    status_code                                                      AS status_key,
    age_band_key,
    CAST(date_format(window_start, 'yyyyMMdd') AS INT)               AS window_start_date_key,
    CAST(date_format(window_end,   'yyyyMMdd') AS INT)               AS snapshot_date_key,
    SUM(exposure_days)                                               AS exposure_days,
    ROUND(SUM(exposure_days) / 365.25, 6)                            AS exposure_years,
    SUM(deregistrations)                                             AS deregistrations
FROM banded
GROUP BY acn, company_type, company_class, company_sub_class, status_code,
         age_band_key, window_start, window_end
"""
df = spark.sql(query)

In [0]:
df.limit(10).display()

In [0]:
(df.write
   .mode("overwrite")
   .format("delta")
   .option("overwriteSchema", "true")
   .saveAsTable(f"{GOLD_SCHEMA}.fct_exposure"))

In [0]:
%sql
SELECT
    COUNT(*)                                                       AS fact_rows,
    COUNT(DISTINCT f.company_key, f.age_band_key, f.snapshot_date_key) AS distinct_grain,  -- must equal fact_rows
    COUNT_IF(c.company_key IS NULL)                                AS orphan_company,        -- expect 0
    COUNT_IF(k.classification_key IS NULL)                         AS orphan_classification, -- expect 0
    COUNT_IF(s.status_key IS NULL)                                 AS orphan_status,         -- expect 0
    COUNT_IF(a.age_band_key IS NULL)                               AS orphan_age_band,       -- expect 0
    COUNT_IF(w.date_key IS NULL)                                   AS orphan_window_start,   -- expect 0
    COUNT_IF(n.date_key IS NULL)                                   AS orphan_snapshot_date   -- expect 0
FROM workspace.sme_risk_gold.fct_exposure f
LEFT JOIN workspace.sme_risk_gold.dim_company                c ON f.company_key           = c.company_key
LEFT JOIN workspace.sme_risk_gold.dim_company_classification k ON f.classification_key    = k.classification_key
LEFT JOIN workspace.sme_risk_gold.dim_status                 s ON f.status_key            = s.status_key
LEFT JOIN workspace.sme_risk_gold.dim_age_band               a ON f.age_band_key          = a.age_band_key
LEFT JOIN workspace.sme_risk_gold.dim_date                   w ON f.window_start_date_key = w.date_key
LEFT JOIN workspace.sme_risk_gold.dim_date                   n ON f.snapshot_date_key     = n.date_key

## Validation – hard stops
Fails the notebook (and the job) if any Gold table breaks its rules: duplicate keys, facts that don't match Silver, keys missing from a dimension, or exposure that doesn't add up.

In [0]:
G, S = GOLD_SCHEMA, SILVER_TABLE

checks = spark.sql(f"""
SELECT
  -- dimensions: one row per key
  (SELECT COUNT(*) - COUNT(DISTINCT company_key)        FROM {G}.dim_company)                AS dim_company_dup_keys,
  (SELECT COUNT(*) - COUNT(DISTINCT classification_key) FROM {G}.dim_company_classification) AS dim_class_dup_keys,
  (SELECT COUNT_IF(borrower_segment = 'Unclassified')   FROM {G}.dim_company_classification) AS unclassified_segments,
  (SELECT COUNT(*) - COUNT(DISTINCT status_key)         FROM {G}.dim_status)                 AS dim_status_dup_keys,
  (SELECT COUNT(*) - COUNT(DISTINCT age_band_key)       FROM {G}.dim_age_band)               AS dim_age_band_dup_keys,
  (SELECT COUNT(*) - COUNT(DISTINCT date_key)           FROM {G}.dim_date)                   AS dim_date_dup_keys,

  -- fct_company_snapshot: one row per Silver company, every key found in its dimension
  (SELECT COUNT(*) FROM {S}) - (SELECT COUNT(*) FROM {G}.fct_company_snapshot)               AS snapshot_vs_silver_rows,
  (SELECT COUNT(*) - COUNT(DISTINCT company_key) FROM {G}.fct_company_snapshot)              AS snapshot_dup_grain,
  (SELECT COUNT(*) FROM {G}.fct_company_snapshot f LEFT ANTI JOIN {G}.dim_company c                ON f.company_key = c.company_key)               AS snapshot_orphan_company,
  (SELECT COUNT(*) FROM {G}.fct_company_snapshot f LEFT ANTI JOIN {G}.dim_company_classification c ON f.classification_key = c.classification_key) AS snapshot_orphan_classification,
  (SELECT COUNT(*) FROM {G}.fct_company_snapshot f LEFT ANTI JOIN {G}.dim_status c                 ON f.status_key = c.status_key)                 AS snapshot_orphan_status,
  (SELECT COUNT(*) FROM {G}.fct_company_snapshot f LEFT ANTI JOIN {G}.dim_age_band c               ON f.age_band_key = c.age_band_key)             AS snapshot_orphan_age_band,
  (SELECT COUNT(*) FROM {G}.fct_company_snapshot f LEFT ANTI JOIN {G}.dim_date c                   ON f.registration_date_key = c.date_key)        AS snapshot_orphan_reg_date,
  (SELECT COUNT(*) FROM {G}.fct_company_snapshot f LEFT ANTI JOIN {G}.dim_date c                   ON f.deregistration_date_key = c.date_key)      AS snapshot_orphan_dereg_date,
  (SELECT COUNT_IF(age_years < 0) FROM {G}.fct_company_snapshot)                                                                                  AS snapshot_negative_age,

  -- fct_exposure: one row per company and age band, keys found, totals add up
  (SELECT COUNT(*) - COUNT(DISTINCT company_key, age_band_key) FROM {G}.fct_exposure)                                                             AS exposure_dup_grain,
  (SELECT COUNT(*) FROM {G}.fct_exposure f LEFT ANTI JOIN {G}.dim_company c                ON f.company_key = c.company_key)                       AS exposure_orphan_company,
  (SELECT COUNT(*) FROM {G}.fct_exposure f LEFT ANTI JOIN {G}.dim_company_classification c ON f.classification_key = c.classification_key)         AS exposure_orphan_classification,
  (SELECT COUNT(*) FROM {G}.fct_exposure f LEFT ANTI JOIN {G}.dim_age_band c               ON f.age_band_key = c.age_band_key)                     AS exposure_orphan_age_band,
  (SELECT COUNT_IF(exposure_days < 0) FROM {G}.fct_exposure)                                                                                      AS exposure_negative_days,
  (SELECT COUNT_IF(is_deregistered AND registration_date IS NOT NULL) FROM {S})
    - (SELECT SUM(deregistrations) FROM {G}.fct_exposure)                                                                                         AS deregistration_mismatch
""").first().asDict()

for k, v in checks.items():
    print(f"{'✅' if v == 0 else '❌'} {k}: {v:,}")

bad = {k: v for k, v in checks.items() if v != 0}
assert not bad, f"Gold validation failed: {bad}"
print("All Gold checks passed")

In [0]:
def q(text):                                   # escape single quotes for SQL
    return text.replace("'", "\\'")

GOLD_COMMENTS = {
    "dim_company": (
        "Company dimension: one row per company (ACN) from the latest ASIC snapshot. Descriptive attributes only. Rebuilt from Silver each run.",
        {
            "company_key": "Primary key: ACN as a number.",
            "acn": "9-digit Australian Company Number as text (keeps leading zeros). ARBN for registrable bodies.",
            "company_name": "Current registered company name (cleaned in Silver).",
            "registration_date": "Date the company was registered. Null for 4 companies.",
            "abn": "ABN recorded by ASIC. Null when none recorded. Not a check of ABN active status.",
            "has_abn": "True when an ABN is recorded. Indicates trading vs dormant; compare within a segment.",
            "is_acn_named": "True when the company own ACN appears in its name (numbered company).",
            "former_name_count": "Number of previous names on the ASIC register.",
            "has_name_change": "True when the company has changed name at least once.",
            "previous_state": "State of registration before the 1991 national scheme. Not current location.",
            "previous_state_label": "Full name of previous_state (NFK = Norfolk Island).",
            "state_registration_number": "Original state registry number. May contain letters.",
            "company_name_has_export_artifact": "Name contains º, which appears to replace | in the ASIC export.",
            "as_of_snapshot_date": "Date of the ASIC file this row comes from.",
        },
    ),
    "dim_company_classification": (
        "Classification dimension: one row per company type + class + sub-class combination, with ASIC labels and the borrower segment used in reporting.",
        {
            "classification_key": "Primary key: type-class-sub_class codes joined, e.g. APTY-LMSH-PROP.",
            "company_type": "ASIC company type code (APTY, APUB, FNOS, RACN, CCIV).",
            "company_type_label": "Description of company_type from the ASIC help file.",
            "company_class": "ASIC liability class code (LMSH, LMGT, LMSG, NLIA, UNLM, NONE).",
            "company_class_label": "Description of company_class from the ASIC help file.",
            "company_sub_class": "ASIC sub-class code (e.g. PROP, PSTC, ULSN, HUNT).",
            "company_sub_class_label": "Description of company_sub_class from the ASIC help file.",
            "borrower_segment": "Reporting segment: Trading company, Super trustee, Not-for-profit, Home unit, Other public, Foreign company, Registrable body, CCIV.",
            "segment_sort_order": "Display order of borrower_segment (1 = Trading company).",
        },
    ),
    "dim_status": (
        "Status dimension: one row per ASIC status code, with flags and a reporting group.",
        {
            "status_key": "Primary key: ASIC status code.",
            "status_code": "ASIC status code (REGD, DRGD, SOFF, EXAD, NOAC, DISS).",
            "status_label": "Description of status_code from the ASIC help file.",
            "is_active": "True for REGD, SOFF, EXAD, NOAC (still on the register).",
            "is_distressed": "True for SOFF (strike-off action) and EXAD (external administration).",
            "is_deregistered": "True for DRGD.",
            "in_population": "True unless CNCL or DISS (excluded from analysis).",
            "status_group": "Reporting group: Registered, Distressed, Not active, Deregistered, Excluded.",
            "status_sort_order": "Display order of statuses.",
        },
    ),
    "dim_age_band": (
        "Age band dimension: fixed company age bands used for survival KPIs. Age counted in full years from registration date.",
        {
            "age_band_key": "Primary key: 1-7 for bands 0-1 to 10+ years; 0 = Unknown (no registration date).",
            "age_band_label": "Band label, e.g. 0-1 years.",
            "min_age_years": "Lowest age in the band (included).",
            "max_age_years": "Upper age limit of the band (excluded). Null for 10+ and Unknown.",
            "age_group": "Wider group: New (under 5 years), Established (5-10 years), Mature (10+ years), Unknown.",
            "is_first_year": "True for the 0-1 years band.",
        },
    ),
    "dim_date": (
        "Date dimension: one row per day from 1 January of the earliest registration year to 31 December of the snapshot year, plus a No date row (key 0).",
        {
            "date_key": "Primary key: date as YYYYMMDD number. 0 = No date.",
            "full_date": "Calendar date.",
            "year": "Calendar year.",
            "quarter": "Calendar quarter (1-4).",
            "month": "Month number (1-12).",
            "month_name": "Month name, e.g. June.",
            "year_month": "Year and month as YYYY-MM.",
            "month_start_date": "First day of the month.",
            "day_of_month": "Day of the month (1-31).",
            "day_name": "Weekday name.",
            "is_weekend": "True for Saturday and Sunday.",
            "financial_year": "Australian financial year (July-June), named by the year it ends, e.g. 2026.",
            "financial_year_label": "Financial year label, e.g. FY2026.",
            "financial_quarter": "Quarter of the Australian financial year (Q1 = Jul-Sep).",
            "is_financial_year_end": "True on 30 June.",
        },
    ),
    "fct_company_snapshot": (
        "Company snapshot fact: one row per company for the latest ASIC snapshot, with status, age band and dates. Rebuilt from Silver each run.",
        {
            "company_key": "Foreign key to dim_company.",
            "classification_key": "Foreign key to dim_company_classification.",
            "status_key": "Foreign key to dim_status.",
            "age_band_key": "Foreign key to dim_age_band. Age at the snapshot date, or at deregistration for deregistered companies.",
            "registration_date_key": "Foreign key to dim_date: registration date (0 = no date).",
            "deregistration_date_key": "Foreign key to dim_date: deregistration date (0 = not deregistered).",
            "snapshot_date_key": "Foreign key to dim_date: date of the ASIC file.",
            "age_years": "Company age in full years (see age_band_key).",
            "company_count": "Always 1. Sum to count companies.",
            "is_active": "1 if status is REGD, SOFF, EXAD or NOAC.",
            "is_distressed": "1 if status is SOFF or EXAD.",
            "is_deregistered": "1 if status is DRGD.",
            "in_population": "1 unless CNCL or DISS.",
        },
    ),
    "fct_exposure": (
        "Exposure fact: one row per company and age band inside the observation window (earliest deregistration date in the file to the snapshot date). Time at risk and deregistrations for survival KPIs.",
        {
            "company_key": "Foreign key to dim_company.",
            "classification_key": "Foreign key to dim_company_classification.",
            "status_key": "Foreign key to dim_status (current status).",
            "age_band_key": "Foreign key to dim_age_band: the age band this time at risk falls in.",
            "window_start_date_key": "Foreign key to dim_date: start of the observation window.",
            "snapshot_date_key": "Foreign key to dim_date: end of the observation window (snapshot date).",
            "exposure_days": "Days at risk in this age band inside the window.",
            "exposure_years": "exposure_days / 365.25 (company-years at risk).",
            "deregistrations": "1 if the company deregistered while in this age band, otherwise 0.",
        },
    ),
}

for table, (table_comment, columns) in GOLD_COMMENTS.items():
    full = f"{GOLD_SCHEMA}.{table}"
    spark.sql(f"COMMENT ON TABLE {full} IS '{q(table_comment)}'")
    for col, text in columns.items():
        spark.sql(f"ALTER TABLE {full} ALTER COLUMN {col} COMMENT '{q(text)}'")
    print(f"Comments applied: {table} ({len(columns)} columns)")